# Swahili News Topic Classification: Data + Baseline

Starter notebook for the African-language text classification assignment.

**What this notebook covers:** loading the data, inspecting it, cleaning, splitting into train/validation/test, a TF-IDF + logistic regression baseline, metrics, error analysis, and saving the model for the web app.

**Next notebooks:** BiGRU/BiLSTM with fastText vectors, then AfroXLMR / AfriBERTa fine-tuning. Reuse the same splits so the comparison is fair.

## 1. Setup

In [1]:
!pip -q install datasets scikit-learn pandas matplotlib seaborn joblib

zsh:1: command not found: pip


In [3]:
import re, random, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ModuleNotFoundError: No module named 'datasets'

## 2. Load the data

The Swahili news dataset has a `train` and a `test` split. We keep the official test set untouched and carve a validation set out of train.

If the first load fails (dataset hosting changes over time), the fallback tries the original name. Check the dataset card on Hugging Face for the exact class counts and cite them in your report.

In [ ]:
try:
    ds = load_dataset("community-datasets/swahili_news")
except Exception as e:
    print("First attempt failed:", e)
    ds = load_dataset("swahili_news")

print(ds)
label_names = ds["train"].features["label"].names
print("Classes:", label_names)

In [ ]:
train_full = ds["train"].to_pandas()
test_df    = ds["test"].to_pandas()

# Map integer labels to readable names
train_full["label_name"] = train_full["label"].map(lambda i: label_names[i])
test_df["label_name"]    = test_df["label"].map(lambda i: label_names[i])

train_full.head()

## 3. Look at the data before touching it

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_full["label_name"].value_counts().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Class distribution (train)")
axes[0].set_ylabel("Articles")

train_full["text"].str.split().str.len().clip(upper=1500).plot.hist(bins=50, ax=axes[1], color="#55A868")
axes[1].set_title("Article length in words (clipped at 1500)")

plt.tight_layout(); plt.show()

print("Duplicate texts in train:", train_full["text"].duplicated().sum())
print("Empty texts in train:", (train_full["text"].str.strip() == "").sum())

## 4. Preprocessing

Keep it light. Swahili is agglutinative (lots of meaning packed into prefixes and suffixes), and aggressive stemming or stopword removal without a proper Swahili tool can hurt more than it helps. What we do:

- lowercase
- remove URLs and email addresses
- remove digits-only tokens and stray symbols
- collapse extra whitespace

Document every step in your report, including the ones you chose *not* to do and why.

In [ ]:
URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
NONWORD  = re.compile(r"[^a-zA-Z\u00C0-\u017F'\s]")   # keep letters and apostrophes
SPACES   = re.compile(r"\s+")

def clean_text(t: str) -> str:
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = NONWORD.sub(" ", t)
    t = SPACES.sub(" ", t).strip()
    return t

for df in (train_full, test_df):
    df["clean"] = df["text"].apply(clean_text)

# Drop duplicates and empties from train only (never alter the official test set)
before = len(train_full)
train_full = train_full[train_full["clean"].str.len() > 0].drop_duplicates(subset="clean").reset_index(drop=True)
print(f"Train: {before} -> {len(train_full)} after removing empties/duplicates")

# Check for leakage between train and test
leak = set(train_full["clean"]) & set(test_df["clean"])
print("Texts appearing in both train and test:", len(leak))

## 5. Splits

- **Test:** the official test split, used once at the very end.
- **Train / validation:** stratified 90/10 split of the official train set.

Think of it as practice questions (train), a mock exam (validation, for tuning), and the real exam (test, touched once).

In [ ]:
train_df, val_df = train_test_split(
    train_full, test_size=0.10, stratify=train_full["label"], random_state=SEED
)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
pd.DataFrame({
    "train": train_df["label_name"].value_counts(normalize=True),
    "val":   val_df["label_name"].value_counts(normalize=True),
    "test":  test_df["label_name"].value_counts(normalize=True),
}).round(3)

## 6. Baseline: TF-IDF + Logistic Regression

This is your tape measure. Every fancier model has to beat these numbers to justify its cost.

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=1000, C=5.0, class_weight="balanced", n_jobs=-1)),
])

baseline.fit(train_df["clean"], train_df["label"])
print("Trained.")

## 7. Evaluation

In [ ]:
def evaluate(model, df, name):
    preds = model.predict(df["clean"])
    acc = accuracy_score(df["label"], preds)
    p, r, f1, _ = precision_recall_fscore_support(df["label"], preds, average="macro", zero_division=0)
    print(f"[{name}] accuracy={acc:.4f}  macro-P={p:.4f}  macro-R={r:.4f}  macro-F1={f1:.4f}")
    return preds

val_preds = evaluate(baseline, val_df, "validation")

In [ ]:
# Run on test ONCE, after you are done tuning on validation
test_preds = evaluate(baseline, test_df, "test")
print()
print(classification_report(test_df["label"], test_preds, target_names=label_names, digits=3))

In [ ]:
cm = confusion_matrix(test_df["label"], test_preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Baseline confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()

## 8. Error analysis

Pull the misclassified test articles, sorted by how confident the model was when it got them wrong. Those are the most interesting ones. Read 30 to 50 of them and tag each with a cause, for example:

- article spans two topics (a minister at a football match)
- English or other-language words mixed in
- very short or low-content text
- label looks questionable
- topic words rare in training data

Put your tags in the `cause` column and count them for the report.

In [ ]:
proba = baseline.predict_proba(test_df["clean"])
errors = test_df.copy()
errors["pred"]       = [label_names[i] for i in test_preds]
errors["confidence"] = proba.max(axis=1)
errors = errors[errors["label"] != test_preds].sort_values("confidence", ascending=False)
errors["cause"] = ""   # fill in by hand after reading

print(f"{len(errors)} misclassified out of {len(test_df)} ({len(errors)/len(test_df):.1%})")

pd.set_option("display.max_colwidth", 300)
errors[["label_name", "pred", "confidence", "text"]].head(15)

In [ ]:
# Which class pairs get confused most?
pairs = errors.groupby(["label_name", "pred"]).size().sort_values(ascending=False).head(10)
pairs

In [ ]:
errors.to_csv("baseline_errors.csv", index=False)   # annotate this file for your report

## 9. Save the baseline for the web app

The Gradio app can load this file straight away, so you have a working demo before the stronger models are ready. Swap in the best model later.

In [ ]:
joblib.dump({"model": baseline, "label_names": label_names, "clean_fn": "clean_text"}, "swahili_baseline.joblib")
print("Saved swahili_baseline.joblib")

## 10. Quick try-out

In [ ]:
def predict(text):
    p = baseline.predict_proba([clean_text(text)])[0]
    top = np.argsort(p)[::-1][:3]
    return [(label_names[i], round(float(p[i]), 3)) for i in top]

predict("Timu ya taifa ya mpira wa miguu imeshinda mechi ya jana usiku")

## Next steps

1. Model 2: fastText Swahili vectors + BiGRU (same splits, same metrics)
2. Model 3: fine-tune AfroXLMR or AfriBERTa on a GPU runtime
3. Comparison table: accuracy, macro-F1, training time, model size
4. Gradio app (`app.py`) loading the best model, deployed on Hugging Face Spaces
5. Report: data source, class definitions, preprocessing, splits, models, metrics, error analysis